In [ ]:
## import dependencies
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import pandas as pd
import torch
import plotly
import requests
#import STalign
#from STalign import STalign
import pickle
import json
import cv2
import scanpy as sc
import anndata as ad
import muon
from scipy import sparse
from torch import nn

# make plots bigger
plt.rcParams["figure.figsize"] = (12,10)

### protein data

In [ ]:
he_id = 'S17_19647_02_05'

In [ ]:
sample_id = he_id + '_RNA'
sample_id

In [ ]:
with open(f'{sample_id}_adata.pickle', 'rb') as handle:
    adata = pickle.load(handle)

### Use the cell embedings from previous results

### run SuperFocus

In [ ]:
import sys
import os

# Get the absolute path to the directory containing the script to be imported
script_dir = os.path.abspath("../SuperFocus")

# Add the directory to sys.path
sys.path.append(script_dir)

In [ ]:
from importlib import reload
import src.model_training.model_training_orig
import src.model_training.processing_image_features
reload(src.model_training.model_training_orig)
reload(src.model_training.processing_image_features)
from src.model_training.model_training_orig import training_orig

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training_orig(adata=adata, pt_model_dir=he_dir, radius=200./2, feature_list=list(adata.var_names), 
              save_model_dir=f'./logs_{sample_id}', hidden_dims=[4096, 4096, 4096, 4096], activations=None, residual=False, norm=False, out_act=None, 
              include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite', kfold=5, cell_embed_dir=f'./{he_id}_cellvit_embed.pickle', sort_way='col_row', datatype='dbit')

In [ ]:
import src.post_process
from importlib import reload
reload(src.post_process)
#reload(src.post_process.post_training_processing)
from src.post_process import post_training_processing

In [ ]:
adata_pred = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}/fold_2/checkpoints/model.best.pth', hidden_dims=[4096, 4096, 4096, 4096], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level1.pickle', feature_list=list(adata_filtered_sub.var_names), datatype='RNA', regress_out=False, resolutions=[0.6, 0.8],
                                               refine_factor=1, sample_id=sample_id, radius=200./2)

In [ ]:
sc.pl.spatial(adata_pred, img_key=None, color='leiden_res_0.6', size=2.0)

In [ ]:
import src.model_training.model_training_refined
reload(src.model_training.model_training_refined)
from src.model_training.model_training_refined import training

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training(adata=adata, pt_model_dir=he_id, radius=200./2, refine_factor=2, 
         feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}/fold_2/checkpoints/model.best.pth',
         save_model_dir=f'./logs_{sample_id}_2x', hidden_dims=[4096, 4096, 4096, 4096], activations=None, residual=False, norm=False, out_act=None, 
         include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite', kfold=3, cell_embed_dir=f'./{he_id}_cellvit_embed.pickle', 
         lr=1e-7, step_size=25, gamma=0.5, num_epochs=50, sort_way='col_row', random_state=42, datatype='dbit', max_fold=1)

In [ ]:
adata_pred_level2 = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_2x/best_model_fold1.pth', hidden_dims=[4096, 4096, 4096, 4096], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level2.pickle', feature_list=list(adata_filtered_sub.var_names), datatype='RNA', regress_out=False, resolutions=[0.6, 0.8],
                                               refine_factor=2, sample_id=sample_id, radius=200./2)

In [ ]:
sc.pl.spatial(adata_pred_level2, img_key=None, color='leiden_res_0.6', size=1.5)

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training(adata=adata, pt_model_dir=he_dir, radius=200./2, refine_factor=4, 
         feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}_2x/best_model_fold1.pth',
         save_model_dir=f'./logs_{sample_id}_4x', hidden_dims=[4096, 4096, 4096, 4096], activations=None, residual=False, norm=False, out_act=None, 
         include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite', kfold=5, cell_embed_dir=f'./{he_id}_cellvit_embed.pickle', 
         lr=1e-8, step_size=25, gamma=0.5, num_epochs=50, sort_way='col_row', random_state=42, datatype='dbit', max_fold=1)

In [ ]:
adata_pred_level4 = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_4x/best_model_fold1.pth', hidden_dims=[4096, 4096, 4096, 4096], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level4.pickle', feature_list=list(adata_filtered_sub.var_names), datatype='protein', regress_out=False, resolutions=[0.4, 0.6],
                                               refine_factor=4, sample_id=sample_id, radius=200./2)

In [ ]:
sc.pl.spatial(adata_pred_level4, img_key=None, color='leiden_res_0.4', size=2.5)

In [ ]:
'''
training(adata=adata_filtered_sub, pt_model_dir=he_dir, radius=200./2, refine_factor=8, 
         feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}_4x/best_model_fold1.pth',
         save_model_dir=f'./logs_{sample_id}_8x', hidden_dims=[4096, 4096, 4096, 4096], activations=None, residual=False, norm=False, out_act=nn.ReLU(), 
         include_tanhshrink=False, sample_id=sample_id, training_mode=None, selected_model='lite', kfold=5, cell_embed_dir=f'./{sample_id}_cellvit_embed.pickle', 
         lr=1e-8, step_size=25, gamma=0.5, num_epochs=50, sort_way='col_row', random_state=42, datatype='dbit')
'''

In [ ]:
'''
adata_pred_level8 = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_8x/best_model_fold1.pth', hidden_dims=[4096, 4096, 4096, 4096], 
                                               activations=None, out_act=nn.ReLU(), residual=False, norm=False, selected_model='lite', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level8.pickle', feature_list=list(adata_filtered_sub.var_names), datatype='protein', regress_out=False, resolutions=[0.6, 0.8],
                                               refine_factor=8, sample_id=sample_id, radius=200./2)
'''

In [ ]:
#sc.pl.spatial(adata_pred_level8, img_key=None, color='leiden_res_0.8', size=4.5)

### extended area

In [ ]:
from src.model_training.processing_image_features import whole_image_all_cells_features
from src.model_training import MLP
import torch

In [ ]:
refine_factor = 4
cell_markers, cell_counts, cell_coords = whole_image_all_cells_features(pt_model_dir=he_dir, radius=100./refine_factor, cell_embed_dir=f'./{he_id}_sub_cellvit_embed.pickle')

In [ ]:
cellvit_marker_array = np.hstack((cell_markers, cell_counts))

In [ ]:
cellvit_marker_array.shape

In [ ]:
with open(f'./{sample_id}_scaler_refine_level{refine_factor}.pickle', 'rb') as file:
    CODEX_HIST_scaler = pickle.load(file)
    
with open(f'./{sample_id}_gene_exp_pd.pickle', 'rb') as file:
    dbit_pd = pickle.load(file)

In [ ]:
codex_cell_marker_scaled = CODEX_HIST_scaler.transform(cellvit_marker_array)

In [ ]:
hidden_dims=[4096, 4096, 4096, 4096]
activations=None
residual=False
norm=False
out_act=None
model = MLP.Model_lite_shrink(n_inp=3609, n_out=3005, hidden_dims=hidden_dims, activations=activations, out_activation=out_act)

In [ ]:
state_dict = torch.load(f'./logs_{sample_id}_{refine_factor}x/best_model_fold1.pth', map_location=torch.device('cpu')) # Load on CPU
model.load_state_dict(state_dict)

In [ ]:
model.eval()

with torch.no_grad():
    output_full = model(torch.tensor(codex_cell_marker_scaled).to(torch.float32))

In [ ]:
predicted_gene= np.array(output_full) * cell_counts[:, 0][:, None]
predicted_gene = np.maximum(0, predicted_gene)

In [ ]:
## Create adata object
adata = ad.AnnData(predicted_gene)#ad.AnnData(sparse.csc_matrix(predicted_gene))

#adata.obs_names = list(dbit_pd.index)
gene_list_full = list(dbit_pd.columns)
#gene_list_full.extend(list(dbit_pd2.columns))
adata.var_names = gene_list_full

adata.obs['x'] = np.array(cell_coords['pxl_col_in_fullres'])
adata.obs['y'] = np.array(cell_coords['pxl_row_in_fullres'])
adata.obs['section_id'] = [he_id]*adata.n_obs

adata.layers['counts'] = predicted_gene.copy()

In [ ]:
adata.uns = {}
adata.uns['spatial'] = {}
adata.uns['spatial'][sample_id] = {}
adata.uns['spatial'][sample_id]['images'] = {}
#adata_query.uns['spatial'][sample_id]['images']['lowres'] = imageLN_flip # rotated_imageLN
adata.uns['spatial'][sample_id]['scalefactors'] = {}
adata.uns['spatial'][sample_id]['scalefactors']['tissue_hires_scalef'] = 16.0  # 78.5
adata.uns['spatial'][sample_id]['scalefactors']['spot_diameter_fullres'] =  20.0 #2.18
adata.uns['spatial'][sample_id]['scalefactors']['fiducial_diameter_fullres'] = 4.24
adata.uns['spatial'][sample_id]['scalefactors']['tissue_lowres_scalef'] = 16.0 # 78.5

adata.obsm['spatial'] = np.array([adata.obs['x'], adata.obs['y']]).T
adata.obs['sample_id'] = [he_id] * adata.n_obs

#sc.pp.calculate_qc_metrics(adata, inplace=True, log1p=True)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
#sc.pp.highly_variable_genes(adata, n_top_genes=2000)

# scale data, clip values exceeding standard deviation 10.
sc.pp.scale(adata, max_value=10)

sc.tl.pca(adata, svd_solver='arpack')
# plot more PCS
sc.pl.pca(adata, components = ['1,2','3,4','5,6','7,8'], ncols=2)

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True, n_pcs = 50)

In [ ]:
# Then we calculate a new umap
sc.pp.neighbors(adata, n_neighbors=100, n_pcs=25, use_rep='X_pca', metric='cosine', random_state=0)

sc.tl.umap(adata, min_dist=0.0, random_state=0)

In [ ]:
sc.tl.leiden(adata, resolution = 0.1, key_added = "leiden_0.1", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(adata, resolution = 0.15, key_added = "leiden_0.15", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(adata, resolution = 0.2, key_added = "leiden_0.2", random_state=0, flavor='igraph', n_iterations=2)

In [ ]:
adata.write_h5ad(f'{sample_id}_adata.h5ad')